In [1]:
import os
import copy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [4]:
drive_root = "/content/drive/MyDrive"

pos_matches = []
for root, dirs, files in os.walk(drive_root):
    needed = {
        "scenario23_pos_beam_train.csv",
        "scenario23_pos_beam_val.csv",
        "scenario23_pos_beam_test.csv"
    }
    if needed.issubset(set(files)):
        pos_matches.append(root)

print("Position CSV folder candidates:")
for p in pos_matches:
    print(p)

Position CSV folder candidates:
/content/drive/MyDrive/Pos beam


In [5]:
POS_ROOT = "/content/drive/MyDrive/Pos beam"

pos_train_csv = os.path.join(POS_ROOT, "scenario23_pos_beam_train.csv")
pos_val_csv   = os.path.join(POS_ROOT, "scenario23_pos_beam_val.csv")
pos_test_csv  = os.path.join(POS_ROOT, "scenario23_pos_beam_test.csv")

print(os.path.exists(pos_train_csv), pos_train_csv)
print(os.path.exists(pos_val_csv), pos_val_csv)
print(os.path.exists(pos_test_csv), pos_test_csv)

True /content/drive/MyDrive/Pos beam/scenario23_pos_beam_train.csv
True /content/drive/MyDrive/Pos beam/scenario23_pos_beam_val.csv
True /content/drive/MyDrive/Pos beam/scenario23_pos_beam_test.csv


In [6]:
train_pos_df = pd.read_csv(pos_train_csv)
val_pos_df   = pd.read_csv(pos_val_csv)
test_pos_df  = pd.read_csv(pos_test_csv)

print("Train:", train_pos_df.shape)
print("Val  :", val_pos_df.shape)
print("Test :", test_pos_df.shape)

print(train_pos_df.head())
print(train_pos_df.columns.tolist())

Train: (6832, 3)
Val  : (3416, 3)
Test : (1139, 3)
   index                                  unit2_pos  unit1_beam
0   3532    [0.8092883966431671, 0.521083920903955]          17
1   2224  [0.4816276084988933, 0.29434536152734486]          14
2   9416    [0.220278556834608, 0.4136596156292844]          17
3   8510  [0.21412273613497904, 0.4547214157104936]          20
4   6877  [0.14500641727379412, 0.4097884695072434]          17
['index', 'unit2_pos', 'unit1_beam']


In [7]:
label_col = train_pos_df.columns[-1]
feature_cols = [c for c in train_pos_df.columns if c != label_col]

print("Feature columns:", feature_cols)
print("Label column:", label_col)

print("Label min/max:", train_pos_df[label_col].min(), train_pos_df[label_col].max())

Feature columns: ['index', 'unit2_pos']
Label column: unit1_beam
Label min/max: 2 30


In [10]:
# Convert feature columns to numeric; invalid strings become NaN
train_features_num = train_pos_df[feature_cols].apply(pd.to_numeric, errors="coerce")

# Mean/std for normalization (ignore NaN). Avoid divide-by-zero with std=1.
train_mean = train_features_num.mean()
train_std = train_features_num.std().replace(0, 1).fillna(1)

print("Mean:")
print(train_mean)

print("Std:")
print(train_std)

# Optional: quick visibility into problematic non-numeric columns
bad_cols = train_features_num.columns[train_features_num.isna().all()].tolist()
if bad_cols:
    print("Non-numeric columns skipped in stats:", bad_cols)

Mean:
index        5666.391247
unit2_pos            NaN
dtype: float64
Std:
index        3288.550573
unit2_pos       1.000000
dtype: float64
Non-numeric columns skipped in stats: ['unit2_pos']


In [15]:
class PositionBeamDataset(Dataset):
    def __init__(self, csv_path, feature_cols, label_col, mean, std):
        self.df = pd.read_csv(csv_path).reset_index(drop=True)
        self.feature_cols = feature_cols
        self.label_col = label_col

        # Keep only features with valid normalization stats (numeric columns).
        valid_cols = [c for c in mean.index if c in self.feature_cols and pd.notna(mean[c])]
        self.mean = mean[valid_cols]
        self.std = std[valid_cols].replace(0, 1).fillna(1)

        # Safely coerce string/object columns to numeric; invalid values become NaN.
        x = self.df[valid_cols].apply(pd.to_numeric, errors="coerce")
        x = x.fillna(self.mean)
        x = (x - self.mean) / self.std

        self.x = torch.tensor(x.values, dtype=torch.float32)
        self.y = torch.tensor(self.df[self.label_col].values, dtype=torch.long)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

In [16]:
batch_size = 64

dataset_pos_train = PositionBeamDataset(pos_train_csv, feature_cols, label_col, train_mean, train_std)
dataset_pos_val   = PositionBeamDataset(pos_val_csv, feature_cols, label_col, train_mean, train_std)
dataset_pos_test  = PositionBeamDataset(pos_test_csv, feature_cols, label_col, train_mean, train_std)

train_loader_pos = DataLoader(dataset_pos_train, batch_size=batch_size, shuffle=True)
val_loader_pos   = DataLoader(dataset_pos_val, batch_size=batch_size, shuffle=False)
test_loader_pos  = DataLoader(dataset_pos_test, batch_size=batch_size, shuffle=False)

print("Train size:", len(dataset_pos_train))
print("Val size  :", len(dataset_pos_val))
print("Test size :", len(dataset_pos_test))

Train size: 6832
Val size  : 3416
Test size : 1139


In [17]:
x_batch, y_batch = next(iter(train_loader_pos))

print("x_batch shape:", x_batch.shape)
print("y_batch shape:", y_batch.shape)
print("First x:", x_batch[:5])
print("First y:", y_batch[:10])

input_dim = x_batch.shape[1]
num_classes = 64

print("input_dim:", input_dim)
print("num_classes:", num_classes)

x_batch shape: torch.Size([64, 1])
y_batch shape: torch.Size([64])
First x: tensor([[-1.2721],
        [ 1.4920],
        [ 1.5604],
        [ 0.1586],
        [-1.4050]])
First y: tensor([17,  4,  2, 26, 15, 20, 15, 30, 11, 17])
input_dim: 1
num_classes: 64


4 layer mlp 

In [18]:
class FourLayerMLP(nn.Module):
    def __init__(self, input_dim, num_classes=64, hidden_dims=(128, 256, 128, 64), dropout=0.2):
        super().__init__()
        h1, h2, h3, h4 = hidden_dims

        self.net = nn.Sequential(
            nn.Linear(input_dim, h1),
            nn.BatchNorm1d(h1),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(h1, h2),
            nn.BatchNorm1d(h2),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(h2, h3),
            nn.BatchNorm1d(h3),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(h3, h4),
            nn.BatchNorm1d(h4),
            nn.ReLU(),

            nn.Linear(h4, num_classes)
        )

    def forward(self, x):
        return self.net(x)

ann 

In [19]:
class SimpleANN(nn.Module):
    def __init__(self, input_dim, num_classes=64, hidden_dim=128, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        return self.net(x)

ann + attention

In [20]:
class ANNWithAttention(nn.Module):
    def __init__(self, input_dim, num_classes=64, hidden_dim=128, dropout=0.2):
        super().__init__()

        self.attn = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, input_dim),
            nn.Softmax(dim=1)
        )

        self.classifier = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        weights = self.attn(x)
        x_weighted = x * weights
        return self.classifier(x_weighted)

rnn 

In [21]:
class PositionRNN(nn.Module):
    def __init__(self, input_dim, num_classes=64, hidden_dim=64, num_layers=2, dropout=0.1):
        super().__init__()

        self.rnn = nn.RNN(
            input_size=1,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x_seq = x.unsqueeze(-1)
        out, h = self.rnn(x_seq)
        last = out[:, -1, :]
        return self.fc(last)

lstm 

In [22]:
class PositionLSTM(nn.Module):
    def __init__(self, input_dim, num_classes=64, hidden_dim=64, num_layers=2, dropout=0.1):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=1,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x_seq = x.unsqueeze(-1)
        out, (h, c) = self.lstm(x_seq)
        last = out[:, -1, :]
        return self.fc(last)

top - k evaluation

In [23]:
def evaluate_topk_pos(model, loader, device, ks=(1, 2, 3, 5)):
    model.eval()
    total = 0
    correct = {k: 0 for k in ks}

    with torch.no_grad():
        for x, labels in loader:
            x = x.to(device)
            labels = labels.to(device)

            outputs = model(x)

            max_k = max(ks)
            _, pred = torch.topk(outputs, k=max_k, dim=1)
            pred = pred.t()

            total += labels.size(0)

            for k in ks:
                correct[k] += pred[:k].eq(labels.view(1, -1)).sum().item()

    return {f"top{k}": 100.0 * correct[k] / total for k in ks}

trainer 

In [24]:
def train_position_model(
    model,
    train_loader,
    val_loader,
    device,
    epochs=50,
    lr=1e-3,
    weight_decay=1e-4,
    milestones=(20, 35),
    save_path="/content/drive/MyDrive/best_position_model.pth"
):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=list(milestones), gamma=0.1)

    best_top1 = -1
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        total_train = 0

        for x, labels in train_loader:
            x = x.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(x)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            bs = labels.size(0)
            running_loss += loss.item() * bs
            total_train += bs

        scheduler.step()

        train_loss = running_loss / total_train
        val_metrics = evaluate_topk_pos(model, val_loader, device, ks=(1, 2, 3, 5))

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            **val_metrics
        }
        history.append(row)

        print(
            f"Epoch {epoch:02d} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Top1: {val_metrics['top1']:.2f} | "
            f"Top2: {val_metrics['top2']:.2f} | "
            f"Top3: {val_metrics['top3']:.2f} | "
            f"Top5: {val_metrics['top5']:.2f}"
        )

        if val_metrics["top1"] > best_top1:
            best_top1 = val_metrics["top1"]
            torch.save(copy.deepcopy(model.state_dict()), save_path)
            print("Saved best model")

    return pd.DataFrame(history)

model builder 

In [25]:
def build_position_model(model_name, input_dim, num_classes=64):
    if model_name == "mlp4":
        return FourLayerMLP(input_dim=input_dim, num_classes=num_classes).to(device)

    elif model_name == "ann":
        return SimpleANN(input_dim=input_dim, num_classes=num_classes).to(device)

    elif model_name == "ann_attention":
        return ANNWithAttention(input_dim=input_dim, num_classes=num_classes).to(device)

    elif model_name == "rnn":
        return PositionRNN(input_dim=input_dim, num_classes=num_classes).to(device)

    elif model_name == "lstm":
        return PositionLSTM(input_dim=input_dim, num_classes=num_classes).to(device)

    else:
        raise ValueError("Unknown model_name")

run all the models 

In [26]:
position_model_names = [
    "mlp4",
    "ann",
    "ann_attention",
    "rnn",
    "lstm"
]

all_position_histories = {}
pos_results = []

for model_name in position_model_names:
    print("\n" + "="*80)
    print(f"Training Position Model: {model_name}")
    print("="*80)

    set_seed(42)

    model_pos = build_position_model(model_name, input_dim=input_dim, num_classes=64)
    save_path = f"/content/drive/MyDrive/best_position_{model_name}.pth"

    history_pos = train_position_model(
        model=model_pos,
        train_loader=train_loader_pos,
        val_loader=val_loader_pos,
        device=device,
        epochs=50,
        lr=1e-3,
        weight_decay=1e-4,
        milestones=(20, 35),
        save_path=save_path
    )

    all_position_histories[model_name] = history_pos

    model_pos_eval = build_position_model(model_name, input_dim=input_dim, num_classes=64)
    model_pos_eval.load_state_dict(torch.load(save_path, map_location=device))

    test_metrics_pos = evaluate_topk_pos(model_pos_eval, test_loader_pos, device, ks=(1, 2, 3, 5))

    print(f"{model_name} Test metrics:", test_metrics_pos)

    pos_results.append({
        "Model": model_name,
        "Top-1": test_metrics_pos["top1"],
        "Top-2": test_metrics_pos["top2"],
        "Top-3": test_metrics_pos["top3"],
        "Top-5": test_metrics_pos["top5"],
    })

    del model_pos
    del model_pos_eval
    torch.cuda.empty_cache()

pos_results_df = pd.DataFrame(pos_results)
pos_results_df


Training Position Model: mlp4
Epoch 01 | Train Loss: 3.0019 | Top1: 25.15 | Top2: 41.83 | Top3: 53.45 | Top5: 67.48
Saved best model
Epoch 02 | Train Loss: 2.4701 | Top1: 28.10 | Top2: 43.27 | Top3: 53.63 | Top5: 68.82
Saved best model
Epoch 03 | Train Loss: 2.4196 | Top1: 26.17 | Top2: 41.63 | Top3: 55.53 | Top5: 68.38
Epoch 04 | Train Loss: 2.4005 | Top1: 27.90 | Top2: 42.62 | Top3: 54.13 | Top5: 68.38
Epoch 05 | Train Loss: 2.3934 | Top1: 25.59 | Top2: 41.33 | Top3: 54.19 | Top5: 69.50
Epoch 06 | Train Loss: 2.3897 | Top1: 25.18 | Top2: 41.77 | Top3: 55.18 | Top5: 70.64
Epoch 07 | Train Loss: 2.3851 | Top1: 25.44 | Top2: 43.09 | Top3: 54.04 | Top5: 69.91
Epoch 08 | Train Loss: 2.3709 | Top1: 28.98 | Top2: 44.29 | Top3: 54.95 | Top5: 70.11
Saved best model
Epoch 09 | Train Loss: 2.3708 | Top1: 27.28 | Top2: 41.83 | Top3: 54.48 | Top5: 71.05
Epoch 10 | Train Loss: 2.3701 | Top1: 28.43 | Top2: 41.83 | Top3: 54.89 | Top5: 69.61
Epoch 11 | Train Loss: 2.3705 | Top1: 27.69 | Top2: 41.42 

,Model,Top-1,Top-2,Top-3,Top-5
0,mlp4,26.777875,41.878841,51.273047,70.237050
1,ann,27.568042,42.493415,51.536435,67.866550
2,ann_attention,26.953468,42.054434,52.238806,67.515364
3,rnn,27.304653,41.703248,50.921861,66.812994
4,lstm,28.533802,42.054434,51.712028,66.988586


In [27]:
pos_results_df = pd.DataFrame(pos_results)
pos_results_df = pos_results_df.sort_values(by="Top-1", ascending=False).reset_index(drop=True)
pos_results_df

,Model,Top-1,Top-2,Top-3,Top-5
0,lstm,28.533802,42.054434,51.712028,66.988586
1,ann,27.568042,42.493415,51.536435,67.866550
2,rnn,27.304653,41.703248,50.921861,66.812994
3,ann_attention,26.953468,42.054434,52.238806,67.515364
4,mlp4,26.777875,41.878841,51.273047,70.237050
